# 03 — Backtest Results

Explores what `BarBacktest` produces: the structure of `bt_result.data`, individual
trade log with entry/exit prices, and per-symbol equity curves as a sanity check.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalAnalytics
      (bars)            (signal cols)   (bt_result)       (analytics)
```

In [9]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalAnalytics
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Data → Signal → Backtest

In [10]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)
analytics = SignalAnalytics(bt_result)

df = bt_result.data
print(f"{df.shape[0]:,} bars  |  {df.index.get_level_values('symbol').nunique()} symbols")

2026-05-01 15:28:43.188 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=de1ec4900549


2,193 bars  |  3 symbols


## 2. bt_result Structure

`BarBacktest.run()` returns a `BarBacktestResult` whose `.data` is a `(symbol, timestamp)`
MultiIndex DataFrame. The original signal columns are preserved and the backtest engine
appends execution and equity columns.

| Column | Description |
|---|---|
| `cycle` | `None` (flat), `init` (entry bar), `long` (in trade), `exit` (exit bar) |
| `return_mark_to_close` | Fill-aware daily return — entry on open after `init`, exit on open after `exit` |
| `return_conservative` | Worst-case fill — entry on `init` high, exit on `exit` low |
| `return_net` | MTC return minus round-trip cost (`cost_bps / 10 000` on entry and exit bars) |
| `position_start` | 1 when capital is deployed at bar open |
| `position_end` | 1 when capital is deployed at bar close |
| `strategy_equity_mark_to_close` | Cumulative product of MTC returns (starts at 1.0) |
| `strategy_equity_conservative` | Cumulative product of conservative returns |
| `strategy_equity_net` | Cumulative product of net returns |
| `trade_cycle_id` | Integer ID incrementing on each new trade or flat period |

In [11]:
bt_result.data.columns

Index(['open', 'high', 'low', 'close', 'volume', 'ma', 'signal_close',
       'signal_open', 'trade_direction', 'turnover', 'enter', 'exit', 'cycle',
       'signal_age', 'return_mark_to_close', 'return_conservative',
       'return_net', 'position_start', 'position_end',
       'strategy_equity_mark_to_close', 'strategy_equity_conservative',
       'strategy_equity_net', 'trade_cycle_id'],
      dtype='str')

In [12]:
exec_cols = [
    "cycle", "return_mark_to_close", "return_conservative", "return_net",
    "position_start", "position_end",
    "strategy_equity_mark_to_close", "strategy_equity_conservative", "strategy_equity_net",
    "trade_cycle_id",
]
df[exec_cols].head(12)

cycle  return_mark_to_close  return_conservative  \
symbol  timestamp                                                     
BTC-USD 2022-01-01  None                   0.0                  0.0   
        2022-01-02  None                   0.0                  0.0   
        2022-01-03  None                   0.0                  0.0   
        2022-01-04  None                   0.0                  0.0   
        2022-01-05  None                   0.0                  0.0   
        2022-01-06  None                   0.0                  0.0   
        2022-01-07  None                   0.0                  0.0   
        2022-01-08  None                   0.0                  0.0   
        2022-01-09  None                   0.0                  0.0   
        2022-01-10  None                   0.0                  0.0   
        2022-01-11  None                   0.0                  0.0   
        2022-01-12  None                   0.0                  0.0   

                    return_net  position_start  position_end  \
symbol  timestamp                                              
BTC-USD 2022-01-01         0.0               0             0   
        2022-01-02         0.0               0             0   
        2022-01-03         0.0               0             0   
        2022-01-04         0.0               0             0   
        2022-01-05         0.0               0             0   
        2022-01-06         0.0               0             0   
        2022-01-07         0.0               0             0   
        2022-01-08         0.0               0             0   
        2022-01-09         0.0               0             0   
        2022-01-10         0.0               0             0   
        2022-01-11         0.0               0             0   
        2022-01-12         0.0               0             0   

                    strategy_equity_mark_to_close  \
symbol  timestamp                                   
BTC-USD 2022-01-01                            1.0   
        2022-01-02                            1.0   
        2022-01-03                            1.0   
        2022-01-04                            1.0   
        2022-01-05                            1.0   
        2022-01-06                            1.0   
        2022-01-07                            1.0   
        2022-01-08                            1.0   
        2022-01-09                            1.0   
        2022-01-10                            1.0   
        2022-01-11                            1.0   
        2022-01-12                            1.0   

                    strategy_equity_conservative  strategy_equity_net  \
symbol  timestamp                                                       
BTC-USD 2022-01-01                           1.0                  1.0   
        2022-01-02                           1.0                  1.0   
        2022-01-03                           1.0                  1.0   
        2022-01-04                           1.0                  1.0   
        2022-01-05                           1.0                  1.0   
        2022-01-06                           1.0                  1.0   
        2022-01-07                           1.0                  1.0   
        2022-01-08                           1.0                  1.0   
        2022-01-09                           1.0                  1.0   
        2022-01-10                           1.0                  1.0   
        2022-01-11                           1.0                  1.0   
        2022-01-12                           1.0                  1.0   

                    trade_cycle_id  
symbol  timestamp                   
BTC-USD 2022-01-01               1  
        2022-01-02               1  
        2022-01-03               1  
        2022-01-04               1  
        2022-01-05               1  
        2022-01-06               1  
        2022-01-07               1  
        2022-01-08          

In [14]:
sym = "BTC-USD"
bt_result.data.query("symbol == @sym").unstack('symbol').to_clipboard()

## 3. Per-Trade Log

One row per entry-to-exit cycle, with fill prices derived directly from the OHLCV
bars that the backtest used — not the signal-trigger close.

| Column | Fill assumption | Source |
|---|---|---|
| `entry_fill_mtc` | Open of the entry bar | MTC entry fill |
| `entry_fill_conservative` | High of the entry bar | Worst-case entry fill |
| `exit_fill_mtc` | Open of the exit bar | MTC exit fill |
| `exit_fill_conservative` | Low of the exit bar | Worst-case exit fill |

Export to Excel to inspect individual trades alongside the actual fills the
backtest assumed.

In [15]:
trades = analytics.trade_stats()

bars_reset = bt_result.data[["open", "high", "low"]].reset_index()

entry_px = (
    bars_reset.rename(columns={"timestamp": "entry_date"})
    [["symbol", "entry_date", "open", "high"]]
    .rename(columns={"open": "entry_fill_mtc", "high": "entry_fill_conservative"})
)
exit_px = (
    bars_reset.rename(columns={"timestamp": "exit_date"})
    [["symbol", "exit_date", "open", "low"]]
    .rename(columns={"open": "exit_fill_mtc", "low": "exit_fill_conservative"})
)

trade_log = (
    trades
    .merge(entry_px, on=["symbol", "entry_date"], how="left")
    .merge(exit_px,  on=["symbol", "exit_date"],  how="left")
)

cols = [
    "symbol", "entry_date", "exit_date", "duration",
    "entry_fill_mtc", "entry_fill_conservative",
    "exit_fill_mtc", "exit_fill_conservative",
    "return_net", "max_intra_drawdown_net",
    "return_conservative", "max_intra_drawdown_conservative",
]

trade_log[cols].round(4)

C:\Users\Dalva\AppData\Local\Temp\ipykernel_28484\3791951790.py:30: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  trade_log[cols].round(4)


,symbol,entry_date,exit_date,duration,entry_fill_mtc,entry_fill_conservative,exit_fill_mtc,exit_fill_conservative,return_net,max_intra_drawdown_net,return_conservative,max_intra_drawdown_conservative
0,BTC-USD,2023-01-14,2023-08-18,217,19910.5371,21075.1426,26636.0781,25668.9219,0.3378,-0.1870,0.2180,-0.1870
1,BTC-USD,2023-08-30,2023-08-31,2,27726.0840,27760.1602,27301.9297,25752.9297,-0.0153,-0.0155,-0.0723,-0.0723
2,BTC-USD,2023-10-17,2024-01-01,77,28522.0977,28618.7520,42280.2344,42214.9766,0.5485,-0.0662,0.5433,-0.0662
3,ETH-USD,2022-01-01,2022-01-08,8,3683.0471,3769.9180,3193.5024,3020.8809,-0.1329,-0.1662,-0.1987,-0.2112
4,ETH-USD,2022-04-04,2022-04-06,3,3522.3650,3535.1482,3411.6721,3171.2051,-0.0314,-0.0314,-0.1029,-0.1029
5,ETH-USD,2023-01-13,2023-08-18,218,1417.9462,1461.6727,1682.0385,1644.9309,0.1862,-0.2215,0.1254,-0.2241
6,ETH-USD,2023-10-26,2023-10-28,3,1787.4819,1865.0952,1780.0842,1773.4366,-0.0041,-0.0133,-0.0491,-0.0491
7,ETH-USD,2023-10-30,2024-01-01,64,1795.5891,1829.2495,2282.8704,2267.0181,0.3101,-0.0865,0.2860,-0.0865
8,SOL-USD,2022-01-01,2022-01-20,20,170.3108,178.9622,135.7913,127.1740,-0.2027,-0.2393,-0.2894,-0.2894
9,SOL-USD,2023-02-21,2023-02-22,2,26.1827,26.4243,24.9498,23.3779,-0.0471,-0.0471,-0.1153,-0.1153


In [16]:
# Export to clipboard — paste into Excel
trade_log[cols].to_clipboard(index=False)
print("Copied to clipboard — paste into Excel")

Copied to clipboard — paste into Excel


## 4. Equity Curves

Per-symbol NAV as a visual sanity check — confirms the signal is deploying and
exiting as expected, and that flat stretches correspond to the signal being off.

**Solid** = MTC (best realistic fill) &nbsp; **Dotted** = Conservative (worst realistic fill).
The gap between them is the fill-risk range for that symbol.

In [17]:
colours = [PALETTE["accent_blue"], PALETTE["accent_green"], PALETTE["accent_purple"]]
eq_mtc  = analytics.equity(method="mtc")
eq_con  = analytics.equity(method="conservative")

fig = go.Figure()
for i, sym in enumerate(eq_mtc.columns):
    c = colours[i % len(colours)]
    fig.add_trace(go.Scatter(x=eq_mtc.index, y=eq_mtc[sym], name=f"{sym} MTC",
                             line=dict(color=c, width=1.8)))
    fig.add_trace(go.Scatter(x=eq_con.index, y=eq_con[sym], name=f"{sym} Conservative",
                             line=dict(color=c, width=1, dash="dot")))

apply_theme(fig, title="Equity Curves — MA-200 Trend Signal (MTC vs Conservative)", height=460)
fig.update_layout(yaxis_title="Equity (normalised to 1.0)")
fig.show()